In [ ]:
import torch
from torchvision.models import resnet18
from torchvision.models.resnet import ResNet18_Weights

import sys
sys.path.append('..')

from models import GenClassifier, classifier_layels_list
from utils.channels import Channels
from utils.dataset import get_datasets, image_transform

In [ ]:
device = 'cuda:2'
train_loader, test_loader = get_datasets(image_transform, 200, 'cifar100')
JSCC_Encoder = resnet18(weights = None)
JSCC_Encoder.fc = torch.nn.Linear(512, 512)
JSCC_Decoder = GenClassifier(512, 100, classifier_layels_list, device)

In [ ]:
JSCC_Encoder.train()
JSCC_Decoder.train()

optimizer_EN = torch.optim.Adam(JSCC_Encoder.parameters(), lr=1e-4)
optimizer_DE = torch.optim.Adam(JSCC_Decoder.parameters(), lr=1e-4)

criterion = torch.nn.CrossEntropyLoss()

for epoch in range(20):
    for i, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer_EN.zero_grad()
        optimizer_DE.zero_grad()

        features = JSCC_Encoder(data)
        output = JSCC_Decoder(features)

        loss = criterion(output, target)
        acc = (output.argmax(dim=1) == target).float().mean()
        loss.backward()

        optimizer_EN.step()
        optimizer_DE.step()

        if i % 10 == 0:
            print(f'Epoch {epoch}, loss: {loss.item():.4f}, acc: {acc.item():.4f}')